# 05 -- Build All Indexes

1. Merge all modality JSONL files into unified metadata
2. Build BM25 index (Pyserini) over text passages
3. Build FAISS text index (BioLORD embeddings)
4. Build FAISS image index (BiomedCLIP embeddings)



In [4]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/multimodal_rag"))
from configs import config
from src.utils import setup_java
setup_java()
print("Java ready")

[16:12:28] INFO utils -- JAVA_HOME=/home/aakul001/.conda/envs/rag310


Java ready


In [15]:
import os
os.environ["JAVA_HOME"] = "/home/aakul001/.conda/envs/rag310/lib/jvm"
os.environ["JVM_PATH"]  = "/home/aakul001/.conda/envs/rag310/lib/jvm/lib/server/libjvm.so"
print("Java env set")

Java env set


## Merge all passages into one metadata file

In [2]:

import json
import pandas as pd
from pathlib import Path
from configs.config import (TEXT_JSONL, IMAGE_JSONL, AUDIO_JSONL, PDF_JSONL,
                             METADATA_PARQUET, INDEXES_DIR)

INDEXES_DIR.mkdir(parents=True, exist_ok=True)

all_records = []
for jsonl_path, name in [
    (TEXT_JSONL,  "text"),
    (IMAGE_JSONL, "images"),
    (AUDIO_JSONL, "audio"),
    (PDF_JSONL,   "pdfs"),
]:
    p = Path(jsonl_path)
    if not p.exists():
        print(f"MISSING: {name}")
        continue
    recs = [json.loads(l) for l in open(p) if l.strip()]
    all_records.extend(recs)
    print(f"Loaded {len(recs):,} from {name}")

df = pd.DataFrame(all_records).drop_duplicates(subset="id")
df.to_parquet(METADATA_PARQUET)
print(f"\nTotal passages: {len(df):,}")
print(df["modality"].value_counts())


Loaded 710,822 from text
Loaded 10,000 from images
Loaded 500 from audio
Loaded 6,275 from pdfs

Total passages: 679,137
modality
text     662362
image     10000
pdf        6275
audio       500
Name: count, dtype: int64


## Build BM25 index

In [12]:
import os
import subprocess
from configs.config import BM25_INPUT_DIR, BM25_INDEX_DIR

jvm_path  = "/home/aakul001/.conda/envs/rag310/lib/jvm/lib/server/libjvm.so"
java_home = "/home/aakul001/.conda/envs/rag310/lib/jvm"

os.environ["JAVA_HOME"] = java_home
os.environ["JVM_PATH"]  = jvm_path

result = subprocess.run([
    "/home/aakul001/.conda/envs/rag310/bin/python",
    "-m", "pyserini.index.lucene",
    "--collection",  "JsonCollection",
    "--input",       str(BM25_INPUT_DIR),
    "--index",       str(BM25_INDEX_DIR),
    "--generator",   "DefaultLuceneDocumentGenerator",
    "--threads",     "4",
    "--storePositions", "--storeDocvectors", "--storeRaw"
], capture_output=True, text=True,
   env={**os.environ,
        "JAVA_HOME": java_home,
        "JVM_PATH":  jvm_path})

print(result.stdout[-1000:] if result.stdout else "No stdout")
print(result.stderr[-500:]  if result.stderr else "No stderr")
print("Exit code:", result.returncode)

) - 560,000 documents indexed
2026-04-26 18:47:25,427 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:307) - Indexing Complete! 669,137 documents indexed
2026-04-26 18:47:25,427 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:308) - ============ Final Counter Values ============
2026-04-26 18:47:25,428 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:309) - indexed:          669,137
2026-04-26 18:47:25,428 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:310) - unindexable:            0
2026-04-26 18:47:25,428 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:311) - empty:                  0
2026-04-26 18:47:25,428 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:312) - skipped:                0
2026-04-26 18:47:25,428 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:313) - errors:                 0
2026-04-26 18:47:25,439 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:316) - Total 669,137 documents indexed in 00:03:49


## Build text FAISS index with BioLORD

In [6]:
import numpy as np
import faiss
import json
from sentence_transformers import SentenceTransformer
from configs.config import (METADATA_PARQUET, FAISS_TEXT_PATH, FAISS_IDS_TEXT,
                             TEXT_EMBED_MODEL, EMBED_BATCH_SIZE)

df = pd.read_parquet(METADATA_PARQUET)
txt_df = df[df["modality"].isin(["text", "audio", "pdf"])]
print(f"Encoding {len(txt_df):,} passages with {TEXT_EMBED_MODEL}...")

model = SentenceTransformer(TEXT_EMBED_MODEL)
embeds = model.encode(
    txt_df["contents"].tolist(),
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)

index = faiss.IndexFlatIP(embeds.shape[1])
index.add(embeds.astype("float32"))
faiss.write_index(index, str(FAISS_TEXT_PATH))

with open(FAISS_IDS_TEXT, "w") as f:
    json.dump(txt_df["id"].tolist(), f)

print(f"Text FAISS done: {index.ntotal:,} vectors")


Encoding 669,137 passages with FremyCompany/BioLORD-2023-C...


Batches: 100%|██████████| 10456/10456 [2:02:14<00:00,  1.43it/s] 


Text FAISS done: 669,137 vectors


## Build image FAISS index with BiomedCLIP

In [8]:
import torch
import faiss
import json
import numpy as np
import pandas as pd
from PIL import Image
from configs.config import (METADATA_PARQUET, FAISS_IMAGE_PATH, FAISS_IDS_IMAGE)

df     = pd.read_parquet(METADATA_PARQUET)
img_df = df[(df["modality"] == "image") & (df["image_path"].astype(str).str.len() > 0)]
print(f"Encoding {len(img_df):,} images...")

# Install open_clip if needed
import subprocess
subprocess.run(["pip", "install", "open_clip_torch", "-q"])

import open_clip
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load BiomedCLIP via open_clip
model, _, preprocess = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)
model = model.to(device)
model.eval()
print("BiomedCLIP loaded")

all_embeds = []
valid_ids  = []

rows = img_df.to_dict("records")
batch_size = 32

for i in range(0, len(rows), batch_size):
    batch  = rows[i:i+batch_size]
    images = []
    ids    = []
    for r in batch:
        try:
            img = preprocess(Image.open(r["image_path"]).convert("RGB"))
            images.append(img)
            ids.append(r["id"])
        except:
            pass
    if not images:
        continue

    img_tensor = torch.stack(images).to(device)
    with torch.no_grad():
        feats = model.encode_image(img_tensor)
        feats = feats / feats.norm(dim=-1, keepdim=True)

    all_embeds.append(feats.cpu().numpy())
    valid_ids.extend(ids)

    if (i // batch_size + 1) % 20 == 0:
        print(f"  {min(i+batch_size, len(rows))}/{len(rows)} images done")

embeds = np.vstack(all_embeds).astype("float32")
index  = faiss.IndexFlatIP(embeds.shape[1])
index.add(embeds)
faiss.write_index(index, str(FAISS_IMAGE_PATH))

with open(FAISS_IDS_IMAGE, "w") as f:
    json.dump(valid_ids, f)

print(f"Image FAISS done: {index.ntotal:,} vectors")

Encoding 5,000 images...
Using device: cuda
BiomedCLIP loaded
  640/5000 images done
  1280/5000 images done
  1920/5000 images done
  2560/5000 images done
  3200/5000 images done
  3840/5000 images done
  4480/5000 images done
Image FAISS done: 5,000 vectors


##  Verify

In [16]:
from src.utils import check_paths
from configs.config import METADATA_PARQUET, BM25_INDEX_DIR, FAISS_TEXT_PATH, FAISS_IMAGE_PATH

print("Index status:")
check_paths(METADATA_PARQUET, BM25_INDEX_DIR, FAISS_TEXT_PATH, FAISS_IMAGE_PATH)

Index status:
  [OK]  metadata.parquet  (463.4 MB)
  [OK]  bm25_index  
  [OK]  faiss_text.bin  (2055.6 MB)
  [OK]  faiss_image.bin  (10.2 MB)


True